In [ ]:
import pandas as pd
from rapidfuzz import process, fuzz
#from collections import defaultdict

#load datasets
subject_unit = pd.read_csv('../../datasets/subject_unit.csv')
uiuc_tre = pd.read_csv('../../datasets/uiuc-tre-dataset.csv')

#lowercase + dropping duplicates
subject_unit['Target_lower'] = subject_unit['Subject'].str.lower()
uiuc_tre['Class_lower'] = uiuc_tre['unit'].str.lower()   # adjust if needed
subject_unit = subject_unit.drop_duplicates(subset=['Target_lower'], keep='first')
uiuc_tre = uiuc_tre.drop_duplicates(subset=['Class_lower'], keep='first')

# Subject list for matching
subject_list = subject_unit['Subject'].tolist()

# Create final dfs for auto and manual matches
subject_autoMatch = subject_unit.copy()
subject_manualMatch = subject_unit.copy()
subject_autoMatch['variants'] = [[] for _ in range(len(subject_unit))]
subject_manualMatch['variants'] = [[] for _ in range(len(subject_unit))]

# match + append
for idx, class_name in enumerate(uiuc_tre['Class_lower']):
    best_match, score, _ = process.extractOne(
        class_name,
        subject_list,
        scorer=fuzz.token_sort_ratio
    )

    append_index = subject_unit.loc[subject_unit['Subject'] == best_match].index[0]

    if score > 75:  # auto-accept threshold
        subject_autoMatch.at[append_index, 'variants'].append(class_name)
    else:
        subject_manualMatch.at[append_index, 'variants'].append(class_name)

#save
subject_autoMatch.to_csv('auto_matched_grouped.csv', index=False)
subject_manualMatch.to_csv('manual_review_grouped.csv', index=False)

print("Grouped matching complete.")
print(f"Auto groups saved → auto_matched_grouped.csv (rows: {len(subject_autoMatch)})")
print(f"Manual review groups saved → manual_review_grouped.csv (rows: {len(subject_manualMatch)})")


✅ Grouped matching complete.
✔️ Auto groups saved → auto_matched_grouped.csv (rows: 198)
📝 Manual review groups saved → manual_review_grouped.csv (rows: 198)


In [ ]:
#manual matchingggg
subject_autoMatch.loc[len(subject_autoMatch)] = {'Code': 'NA', 'Subject': 'Not Availible', 'variants': []}


# Loop over each row
for idx, row in  subject_manualMatch.iterrows():
    code = row['Code']
    subject = row['Subject']
    variants = row['variants']

    new_variants = []
    for variant in variants:
        if not variants: #check for empty lists (nothing needed to match manually)
            continue
        variant = variant.strip()

        #ask whether variant matches target
        response = input(f"Is '{variant}' correct for '{subject}'? (Y/N): ").strip().upper()
        while response not in ('Y', 'N'):
            response = input("Please type Y or N: ").strip().upper()

        #if yes, add variant to correct-match df
        if response == 'Y':
            subject_autoMatch.at[idx, 'variants'].append(variant)
        else:  #if no, manually add variant to correct code
            response = input(f"Please type '{variant}' code:").strip().upper()
            if response == 'NA':
                continue
            while response not in subject_unit['Code'].values:
                response = input(f"Code not found. Please type a valid code for '{variant}':").strip().upper()

            new_idx = subject_unit.index[subject_unit['Code'] == response][0]
            subject_autoMatch.at[new_idx, 'variants'].append(variant)   
    
    #save periodically because manually labeling is super annoying when it doesn't save..
    if idx % 20 == 0:
        subject_autoMatch.to_csv('auto_matched_grouped.csv', index=True)
        print(f"Progress saved at row {idx}.")

subject_autoMatch.to_csv('auto_matched_grouped.csv', index=True)
# Save the correct variants to a new CSV
print("\nAll done! Correct variants saved to 'correct_variants.csv'")


Progress saved at row 0.
Progress saved at row 20.
Progress saved at row 40.
Progress saved at row 60.
Progress saved at row 80.
Progress saved at row 100.
Progress saved at row 120.
Progress saved at row 140.
Progress saved at row 160.
Progress saved at row 180.

All done! Correct variants saved to 'correct_variants.csv'


In [8]:
import pandas as pd
import ast

# Read CSV
subject_autoMatch = pd.read_csv('auto_matched_grouped.csv')

# Ensure 'variants' column is a list
subject_autoMatch['variants'] = subject_autoMatch['variants'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Add 'Target_lower' to 'variants' if not already present
for idx, row in subject_autoMatch.iterrows():
    target = row['Target_lower']
    variants = row['variants']
    
    # Only append if target is not in variants
    if target not in variants:
        row['variants'].append(target)

# Drop unnecessary columns
subject_autoMatch.drop(columns=['Target_lower', 'Subject'], inplace=True)

# Save to CSV
subject_autoMatch.to_csv('ALL-subject-code.csv', index=False)
